# DP4 Comparative Data Preparation
**Group 10 · Mai Anh · Shinjuku Airbnb**

RQ3: How does Shinjuku's market structure differ from other Tokyo areas under the announced regulatory changes?

This notebook prepares the comparison dataset and audits; substantive interpretation belongs to M4. Run cells in order. Python logic lives in `scripts/dp4_prepare.py` so the notebook and command-line run share one implementation.

The included outputs were computed from the supplied project data. Optional final cluster labels and predictions are not yet available.

## 1. Locate the project and load the preparation module
Paths are relative to this package; personal usernames are not printed.

In [1]:
# 1. Tìm folder dự án mà không hard-code hoặc in đường dẫn cá nhân.
from pathlib import Path
import sys
import importlib
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "scripts" / "dp4_prepare.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open the DP4_MaiAnh folder in VS Code before running this notebook.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
import dp4_prepare
importlib.reload(dp4_prepare)
print("Project located. Inputs and code are ready.")

Project located. Inputs and code are ready.


## 2. Verify the official input cohort
The baseline contains four wards in the same Q2 snapshot. The Q3 index covers all Tokyo listings, not only the four wards.

In [2]:
# 2. Kiểm tra ID, kỳ, số dòng từng quận, nhóm sức chứa và checksum đầu vào.
source, followup = dp4_prepare.load_sources(ROOT / "inputs")
print("Baseline rows:", len(source))
print("Full Tokyo follow-up IDs:", len(followup))
print(source.groupby("neighbourhood_cleansed").size().to_string())

Baseline rows: 14935
Full Tokyo follow-up IDs: 35970
neighbourhood_cleansed
Shibuya Ku     2433
Shinjuku Ku    5689
Taito Ku       3581
Toshima Ku     3232


## 3. Build the comparison table and export the complete DP4 outputs
No baseline rows are removed; original DP1 values and missingness are preserved. No final model results are assumed.

In [3]:
# 3. Chạy pipeline và xuất bộ bảng; cột mô hình chưa có kết quả sẽ để trống.
tables, qa = dp4_prepare.run(ROOT / "inputs", ROOT / "outputs")
print("QA status:", qa["status"])
print("Rows:", qa["rows"], "| Columns:", qa["columns"])
print("Price-missing rows retained:", qa["price_missing_retained"])
print("Duplicate keys:", qa["duplicate_baseline_keys"])
print("Row inflation:", qa["row_inflation"])
print("Label disagreements with DP3:", qa["shinjuku_label_disagreements_with_dp3"])

DP4 PASS | 14,935 rows | 48 group cells | no row inflation
Pending: 5,328 cluster labels; 5,689 predictions. These do not block DP4 preparation.
QA status: PASS
Rows: 14935 | Columns: 42
Price-missing rows retained: 953
Duplicate keys: 0
Row inflation: 0
Label disagreements with DP3: 0


## 4. Review ward coverage and denominators
Prices use observed-price samples; next-snapshot presence uses all listings. Rates are expressed as percentages from 0 to 100.

In [4]:
# 4. Xem cỡ mẫu và số liệu tổng hợp; chỉ làm tròn lúc hiển thị.
summary = tables["DP4_ward_summary"]
print(summary.round(3).to_string(index=False))

neighbourhood_cleansed  n_listings  n_unique_hosts  n_price  n_price_missing  price_coverage_pct  price_median_jpy  price_q25_jpy  price_q75_jpy  n_present_next  n_presence_denominator  presence_pct
           Shinjuku Ku        5689            1173     5328              361              93.654         20552.335        14800.0      33347.968            5372                    5689        94.428
            Shibuya Ku        2433             696     2292              141              94.205         25662.735        17000.0      39052.500            2300                    2433        94.533
            Toshima Ku        3232             789     3000              232              92.822         17950.000        13272.0      27262.500            3065                    3232        94.833
              Taito Ku        3581             826     3362              219              93.884         22000.000        14813.0      32704.000            3360                    3581        93.829


## 5. Review missingness
Missing prices do not exclude listings from composition or presence statistics.

In [5]:
# 5. Chỉ hiển thị biến có thiếu để bảng đọc gọn; CSV vẫn giữ toàn bộ 44 dòng audit.
missing = tables["DP4_missingness"]
print(missing.loc[missing.n_missing.gt(0)].round(2).to_string(index=False))

neighbourhood_cleansed                   variable  n_total  n_observed  n_missing  missing_pct
           Shinjuku Ku              price_numeric     5689        5328        361         6.35
           Shinjuku Ku             bedrooms_clean     5689        5200        489         8.60
           Shinjuku Ku                 beds_clean     5689        5175        514         9.03
           Shinjuku Ku    reviews_per_month_clean     5689        4989        700        12.30
           Shinjuku Ku review_scores_rating_clean     5689        4989        700        12.30
              Taito Ku              price_numeric     3581        3362        219         6.12
              Taito Ku             bedrooms_clean     3581        3338        243         6.79
              Taito Ku                 beds_clean     3581        3223        358        10.00
              Taito Ku            bathrooms_clean     3581        3578          3         0.08
              Taito Ku    reviews_per_month_clean 

## 6. Check support for within-stratum comparisons
The 30-observation threshold is a descriptive caution, not a significance criterion. It is checked independently for all listings and observed prices. No rows are dropped.

In [6]:
# 6. Đếm số nhóm có dữ liệu/đủ mẫu ở cả Shinjuku và từng comparator.
support = tables["DP4_pairwise_support"]
flags = ["both_have_listings", "both_listing_n_ge_threshold", "both_price_n_ge_threshold"]
print(support.groupby("comparator")[flags].sum().to_string())
print("Each comparator has 12 possible room-type x capacity strata.")
print("Inspect DP4_pairwise_support.csv for the exact groups and denominators.")

            both_have_listings  both_listing_n_ge_threshold  both_price_n_ge_threshold
comparator                                                                            
Shibuya Ku                   7                            6                          6
Taito Ku                    12                            6                          6
Toshima Ku                   9                            6                          6
Each comparator has 12 possible room-type x capacity strata.
Inspect DP4_pairwise_support.csv for the exact groups and denominators.


## 7. Inspect pending model-result coverage
Missing model outputs are not zero. Cluster membership applies only to the Shinjuku DP2 cohort; comparator labels are not inferred.

In [7]:
# 7. Kiểm tra kết quả mô hình đang chờ và listing nằm ngoài phạm vi.
print(tables["DP4_join_audit"].to_string(index=False))
print("Cluster labels still needed:", qa["unresolved_rq1_eligible"])
print("Final test predictions still needed:", qa["unresolved_rq2_in_scope"])

neighbourhood_cleansed     rq1_cluster_status      rq2_prediction_status  n_listings
            Shibuya Ku      outside_rq1_scope          outside_rq2_scope        2433
           Shinjuku Ku  awaiting_final_labels awaiting_final_predictions        5328
           Shinjuku Ku excluded_missing_price awaiting_final_predictions         361
              Taito Ku      outside_rq1_scope          outside_rq2_scope        3581
            Toshima Ku      outside_rq1_scope          outside_rq2_scope        3232
Cluster labels still needed: 5328
Final test predictions still needed: 5689


## 8. Integrate final results later
Do not run this step with provisional labels or training-set fitted probabilities. The final RQ2 model must already be frozen. See the exact schemas in README.

Uncomment only after the corresponding files are supplied. Using `outputs_integrated` preserves the original preparation delivery.

In [8]:
# 8. Chỉ mở comment khi đã nhận đúng file kết quả CUỐI từ Toàn/Hoàng.
# integrated_tables, integrated_qa = dp4_prepare.run(
#     ROOT / "inputs",
#     ROOT / "outputs_integrated",
#     clusters=ROOT / "inputs" / "rq1_final_clusters.csv",
#     predictions=ROOT / "inputs" / "rq2_final_test_predictions.csv",
# )
# print(integrated_qa)
print("Optional integration is not executed: final model files have not been supplied.")

Optional integration is not executed: final model files have not been supplied.


## 9. Run checks for preservation, denominators and invalid joins
These checks deliberately attempt duplicate keys, wrong periods, out-of-scope IDs and invalid probabilities.

In [9]:
# 9. Chạy kiểm tra; dừng notebook nếu có lỗi để tránh bàn giao nhầm.
import unittest
import io
suite = unittest.defaultTestLoader.discover(str(ROOT / "tests"), pattern="test_dp4.py")
log = io.StringIO()
result = unittest.TextTestRunner(stream=log, verbosity=1).run(suite)
print(log.getvalue())
if not result.wasSuccessful():
    raise RuntimeError("DP4 checks failed. Resolve before handing over.")

.........
----------------------------------------------------------------------
Ran 9 tests in 0.777s

OK



## 10. Report and handover

Use `docs/DP4_Methods_Report_EN.md` as the methods draft. Use `docs/DP4_Handover_VI.md` to coordinate M4 with An and Toàn.

DP4 is ready when the preparation QA passes. Final integration is separately pending. Descriptive comparisons do not identify regulatory effects, legal compliance or permanent business exit. Do not tune RQ2 using Q3 descriptive outcomes.

**Presentation conventions:** English labels; JPY/night for prices; percentages for rates; percentage points for rate differences. Format counts as integers and rates with one decimal in slides, while retaining full precision in analytical CSVs.